# RepoCoder — 50 essais (dataset officiel + retriever AST + modèle HuggingFace)

Ce notebook est **autonome** : il n'a besoin d'aucun accès au dépôt git local, il réécrit sur place le code de `repocoder-mine` (copié tel quel) puis télécharge le dataset et les dépôts réels officiels de RepoCoder (`microsoft/CodeT`).

Pipeline pour chaque tâche :
1. Découpage AST du dépôt réel de la tâche (`ast_chunker`, mis en cache sur disque).
2. Filtrage anti-fuite (rien après `context_start_lineno` dans le fichier de la tâche).
3. Retrieval top-k par Jaccard sur les identifiants AST (`retriever.retrieve_top_k_ast_jaccard`).
4. Prompt = extraits récupérés + code inachevé.
5. Génération avec un modèle HuggingFace chargé sur le GPU Colab (`generator.call_huggingface_api`, copié tel quel).
6. Score Exact Match / Edit Similarity (même logique que `repocoder/compute_score.py` officiel).

**Runtime requis : GPU** (Modifier > Paramètres du notebook > GPU, ex. T4).

## 0. Vérifier le GPU

In [ ]:
!nvidia-smi

## 1. Installer les dépendances

In [ ]:
!pip install -q scikit-learn editdistance transformers accelerate

## 2. Réécrire le code du projet (copié tel quel depuis `repocoder-mine/`)

`dataset.py` et `ast_chunker.py` sont copiés intégralement (aucune modification). `generator_min.py` ne garde que `call_huggingface_api` de `generator.py` — le reste du fichier (`groq`, `iterate`, `metrics`) n'est pas nécessaire ici et éviterait des imports inutiles.

In [ ]:
%%writefile dataset.py
import json
import os
import random
import re
from pathlib import Path
from typing import Any, Literal


REQUIRED_FIELDS = {"prompt", "groundtruth", "right_context"}
Split = Literal[
    "baseline",
    "bm25",
    "unixcoder",
    "openai",
    "oracle_bm25",
    "oracle_unixcoder",
    "oracle_openai",
]


def load_jsonl(file_path: str | Path) -> list[dict[str, Any]]:
    """Charger les exemples valides d'un fichier JSONL."""
    records = []
    path = Path(file_path)

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                print(f"Ligne ignorée {line_number}: JSON invalide ({error})")
                continue

            if not isinstance(record, dict):
                print(f"Ligne ignorée {line_number}: l'objet n'est pas un dictionnaire")
                continue

            missing_fields = REQUIRED_FIELDS - record.keys()
            if missing_fields:
                print(
                    f"Ligne ignorée {line_number}: champs manquants "
                    f"{sorted(missing_fields)}"
                )
                continue

            if any(not isinstance(record[field], str) for field in REQUIRED_FIELDS):
                print(f"Ligne ignorée {line_number}: un champ contient une valeur invalide")
                continue

            records.append(record)

    return records


def load_cceval_examples(
    path: str | Path | None = None,
    language: str = "python",
    split: Split = "baseline",
    sample: int | None = None,
    seed: int = 42,
):
    """Charger les exemples avec le modèle officiel de CrossCodeEval."""
    from cceval.dataset import load_cceval_dataset as load_official_dataset

    return load_official_dataset(
        path=str(path) if path is not None else None,
        language=language,
        split=split,
        sample=sample,
        seed=seed,
    )


def load_cceval_dataset(
    path: str | Path | None = None,
    language: str = "python",
    split: Split = "baseline",
    sample: int | None = None,
    seed: int = 42,
) -> list[dict[str, Any]]:
    """Charger le dataset CrossCodeEval avec ses champs d'évaluation."""
    if path is None:
        path = resolve_cceval_path(split, language)

    records = load_jsonl(path)
    task_ids = [record.get("metadata", {}).get("task_id") for record in records]
    task_ids = [task_id for task_id in task_ids if task_id is not None]
    if len(task_ids) != len(set(task_ids)):
        raise ValueError("Le dataset contient des task_id en double")

    if sample is not None:
        if sample < 0:
            raise ValueError("sample doit être positif")
        records = random.Random(seed).sample(records, min(sample, len(records)))

    return records


def resolve_cceval_path(split: Split = "baseline", language: str = "python") -> Path:
    """Construire le chemin CrossCodeEval depuis CCEVAL_DATA_DIR."""
    data_dir = os.environ.get("CCEVAL_DATA_DIR")
    if data_dir is None:
        raise ValueError("La variable CCEVAL_DATA_DIR n'est pas définie")

    filenames = {
        "baseline": "line_completion.jsonl",
        "bm25": "line_completion_rg1_bm25.jsonl",
        "unixcoder": "line_completion_rg1_unixcoder_cosine_sim.jsonl",
        "openai": "line_completion_rg1_openai_cosine_sim.jsonl",
        "oracle_bm25": "line_completion_oracle_bm25.jsonl",
        "oracle_unixcoder": "line_completion_oracle_unixcoder_cosine_sim.jsonl",
        "oracle_openai": "line_completion_oracle_openai_cosine_sim.jsonl",
    }
    return Path(data_dir) / language / filenames[split]


SLIDING_WINDOW_SIZE = 20  # S_w dans l'article RepoCoder
SLIDING_STRIDE = 10  # S_s dans l'article RepoCoder


def slide_over_text(
    text: str,
    window_size: int = SLIDING_WINDOW_SIZE,
    stride: int = SLIDING_STRIDE,
) -> list[str]:
    """Découper un texte en fenêtres glissantes de lignes (S_w, S_s de RepoCoder)."""
    if window_size <= 0:
        raise ValueError("window_size doit être supérieur à 0")
    if stride <= 0 or stride > window_size:
        raise ValueError("stride doit être compris entre 1 et window_size")

    lines = text.splitlines(keepends=True)
    if not lines:
        return []

    windows = []
    for start in range(0, len(lines), stride):
        window_text = "".join(lines[start:start + window_size]).strip()
        if window_text:
            windows.append(window_text)
        if start + window_size >= len(lines):
            break
    return windows


def last_lines(text: str, n: int = SLIDING_STRIDE) -> str:
    """Garder les n dernières lignes d'un texte (utilisé comme requête de retrieval)."""
    lines = text.splitlines(keepends=True)
    return "".join(lines[-n:])


def first_lines(text: str, n: int = SLIDING_STRIDE) -> str:
    """Garder les n premières lignes d'un texte (utilisé sur la prédiction précédente)."""
    lines = text.splitlines(keepends=True)
    return "".join(lines[:n])


def extract_repository_snippets(
    records: list[dict[str, Any]],
    window_size: int = SLIDING_WINDOW_SIZE,
    stride: int = SLIDING_STRIDE,
) -> dict[str, list[dict[str, Any]]]:
    """Regrouper les exemples CCEval par dépôt puis extraire leurs fenêtres glissantes.

    Reproduit le découpage de RepoCoder (S_w=20, S_s=10 par défaut) : chaque
    dépôt (metadata.repository) reçoit la liste des morceaux de code obtenus
    en faisant glisser une fenêtre sur les lignes de chaque exemple.
    """
    repositories: dict[str, list[dict[str, Any]]] = {}

    for record in records:
        metadata = record.get("metadata", {})
        repository = metadata.get("repository") if isinstance(metadata, dict) else None
        if not repository:
            continue

        file_content = record["prompt"] + record.get("right_context", "")
        snippets = slide_over_text(file_content, window_size=window_size, stride=stride)

        for snippet in snippets:
            repositories.setdefault(repository, []).append(
                {
                    "task_id": metadata.get("task_id"),
                    "file": metadata.get("file"),
                    "snippet": snippet,
                }
            )

    return repositories


def extract_cceval_repository_snippets(
    path: str | Path | None = None,
    language: str = "python",
    split: Split = "baseline",
    window_size: int = SLIDING_WINDOW_SIZE,
    stride: int = SLIDING_STRIDE,
    sample: int | None = None,
    seed: int = 42,
) -> dict[str, list[dict[str, Any]]]:
    """Charger le dataset CCEval puis extraire les fenêtres glissantes par dépôt.

    Utilise par défaut les hyperparamètres de l'article RepoCoder
    (S_w=20, S_s=10) pour construire la base de code de chaque dépôt.
    """
    records = load_cceval_dataset(
        path=path, language=language, split=split, sample=sample, seed=seed
    )
    return extract_repository_snippets(records, window_size=window_size, stride=stride)


def save_repository_snippets(
    repositories: dict[str, list[dict[str, Any]]], output_dir: str | Path
) -> dict[str, int]:
    """Sauvegarder les fenêtres glissantes de chaque dépôt dans son propre fichier JSONL."""
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    counts = {}
    for repository, snippets in repositories.items():
        safe_name = re.sub(r"[^\w.-]", "_", repository)
        save_jsonl(snippets, output_path / f"{safe_name}.jsonl")
        counts[repository] = len(snippets)

    return counts


def prepare_repository_snippets(
    output_dir: str | Path,
    path: str | Path | None = None,
    language: str = "python",
    window_size: int = SLIDING_WINDOW_SIZE,
    stride: int = SLIDING_STRIDE,
) -> dict[str, int]:
    """Extraire les fenêtres glissantes du split baseline (line_completion.jsonl) et les sauvegarder par dépôt."""
    repositories = extract_cceval_repository_snippets(
        path=path,
        language=language,
        split="baseline",
        window_size=window_size,
        stride=stride,
    )
    return save_repository_snippets(repositories, output_dir)


def split_dataset(
    records: list[dict[str, Any]],
    train_ratio: float = 0.8,
    validation_ratio: float = 0.1,
    seed: int = 42,
) -> dict[str, list[dict[str, Any]]]:
    """Mélanger les exemples puis créer les ensembles train/validation/test."""
    if train_ratio <= 0 or validation_ratio < 0:
        raise ValueError("Les ratios doivent être positifs")
    if train_ratio + validation_ratio >= 1:
        raise ValueError("La somme des ratios doit être inférieure à 1")

    shuffled_records = records.copy()
    random.Random(seed).shuffle(shuffled_records)

    train_end = int(len(shuffled_records) * train_ratio)
    validation_end = train_end + int(len(shuffled_records) * validation_ratio)

    return {
        "train": shuffled_records[:train_end],
        "validation": shuffled_records[train_end:validation_end],
        "test": shuffled_records[validation_end:],
    }


def split_by_repository(
    records: list[dict[str, Any]],
    train_ratio: float = 0.8,
    validation_ratio: float = 0.1,
    seed: int = 42,
) -> dict[str, list[dict[str, Any]]]:
    """Séparer les dépôts entiers pour éviter une fuite entre les ensembles."""
    repositories = {}
    records_without_repository = []

    for record in records:
        metadata = record.get("metadata", {})
        repository = metadata.get("repository") if isinstance(metadata, dict) else None
        if repository:
            repositories.setdefault(repository, []).append(record)
        else:
            records_without_repository.append(record)

    if not repositories:
        return split_dataset(records, train_ratio, validation_ratio, seed)

    repository_names = list(repositories)
    random.Random(seed).shuffle(repository_names)
    train_repository_end = max(1, int(len(repository_names) * train_ratio))
    validation_repository_end = train_repository_end + int(
        len(repository_names) * validation_ratio
    )

    splits = {
        "train": [],
        "validation": [],
        "test": [],
    }
    for repository in repository_names[:train_repository_end]:
        splits["train"].extend(repositories[repository])
    for repository in repository_names[train_repository_end:validation_repository_end]:
        splits["validation"].extend(repositories[repository])
    for repository in repository_names[validation_repository_end:]:
        splits["test"].extend(repositories[repository])

    # Les exemples sans dépôt sont répartis uniquement après le découpage principal.
    splits["train"].extend(records_without_repository)
    return splits


def save_jsonl(records: list[dict[str, Any]], file_path: str | Path) -> None:
    """Sauvegarder une liste d'exemples au format JSONL."""
    path = Path(file_path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(record, ensure_ascii=False) + "\n")


def prepare_dataset(input_path: str | Path, output_dir: str | Path) -> dict[str, int]:
    """Charger, séparer et sauvegarder le dataset dans trois fichiers."""
    records = load_jsonl(input_path)
    splits = split_by_repository(records)
    output_path = Path(output_dir)

    for split_name, split_records in splits.items():
        save_jsonl(split_records, output_path / f"{split_name}.jsonl")

    return {split_name: len(split_records) for split_name, split_records in splits.items()}


if __name__ == "__main__":
    project_dir = Path(__file__).resolve().parent
    source = Path(r"C:\Users\User\Downloads\line_completion.jsonl")
    destination = project_dir / "data"
    counts = prepare_dataset(source, destination)

    print("Dataset organisé :")
    for split_name, count in counts.items():
        print(f"- {split_name}: {count} exemples")

    # Première extraction : fenêtres glissantes (S_w=20, S_s=10) par dépôt,
    # à partir de line_completion.jsonl uniquement, sauvegardées dans un dossier dédié.
    repository_destination = project_dir / "data" / "repositories"
    repository_counts = prepare_repository_snippets(repository_destination, path=source)

    print(f"\nFenêtres glissantes sauvegardées dans {repository_destination} :")
    for repository, count in repository_counts.items():
        print(f"- {repository}: {count} fenêtres")

In [ ]:
%%writefile ast_chunker.py
import ast
import hashlib
import os
import pickle
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from dataset import SLIDING_STRIDE, SLIDING_WINDOW_SIZE

PROJECT_DIR = Path(__file__).resolve().parent
DEFAULT_CACHE_DIR = PROJECT_DIR / "data" / "cache" / "ast_chunks"


@dataclass
class ScopeBlock:
    """Un bloc nommé (classe ou fonction/méthode) avec ses bornes de lignes et ses identifiants propres."""

    kind: str  # "class" ou "function"
    name: str
    line_start: int
    line_end: int
    identifiers: set[str] = field(default_factory=set)


class _LocalNamesCollector(ast.NodeVisitor):
    """Collecte les noms assignés/définis dans un scope, sans descendre dans les classes/fonctions imbriquées
    (elles ont leur propre ScopeBlock)."""

    def __init__(self) -> None:
        self.names: set[str] = set()

    def visit_FunctionDef(self, node: ast.FunctionDef) -> None:
        self.names.add(node.name)

    def visit_AsyncFunctionDef(self, node: ast.AsyncFunctionDef) -> None:
        self.names.add(node.name)

    def visit_ClassDef(self, node: ast.ClassDef) -> None:
        self.names.add(node.name)

    def visit_arg(self, node: ast.arg) -> None:
        self.names.add(node.arg)

    def visit_Name(self, node: ast.Name) -> None:
        if isinstance(node.ctx, ast.Store):
            self.names.add(node.id)

    def visit_ExceptHandler(self, node: ast.ExceptHandler) -> None:
        if node.name:
            self.names.add(node.name)
        self.generic_visit(node)

    def visit_Import(self, node: ast.Import) -> None:
        for alias in node.names:
            self.names.add(alias.asname or alias.name.split(".")[0])

    def visit_ImportFrom(self, node: ast.ImportFrom) -> None:
        for alias in node.names:
            self.names.add(alias.asname or alias.name)


def _collect_local_names(node: ast.AST) -> set[str]:
    """Noms locaux à un bloc (paramètres, variables assignées, imports locaux, fonctions/classes imbriquées)."""
    collector = _LocalNamesCollector()
    collector.generic_visit(node)  # generic_visit: visite les enfants, pas node lui-même
    return collector.names


def _collect_class_attributes(class_node: ast.ClassDef) -> set[str]:
    """Attributs de classe (`x = 1` dans le corps) et d'instance (`self.x = ...` dans les méthodes)."""
    attributes: set[str] = set()

    for stmt in class_node.body:
        if isinstance(stmt, ast.Assign):
            for target in stmt.targets:
                if isinstance(target, ast.Name):
                    attributes.add(target.id)
        elif isinstance(stmt, ast.AnnAssign) and isinstance(stmt.target, ast.Name):
            attributes.add(stmt.target.id)

    for node in ast.walk(class_node):
        if isinstance(node, ast.Attribute) and isinstance(node.ctx, ast.Store) and isinstance(node.value, ast.Name):
            attributes.add(node.attr)

    return attributes


def build_scope_map(code: str) -> tuple[set[str], list[ScopeBlock]]:
    """Analyser le code d'un fichier et retourner (imports du module, blocs classes/fonctions).

    Les imports de haut niveau (pas dans une fonction/classe) sont visibles dans
    tout le fichier. Chaque classe et chaque fonction/méthode devient un
    ScopeBlock avec ses propres identifiants (nom, attributs/paramètres,
    variables locales) et ses bornes de lignes (`node.lineno`/`node.end_lineno`).
    """
    tree = ast.parse(code)
    module_imports: set[str] = set()
    blocks: list[ScopeBlock] = []

    def visit(node: ast.AST, inside_def: bool) -> None:
        for child in ast.iter_child_nodes(node):
            if isinstance(child, (ast.Import, ast.ImportFrom)) and not inside_def:
                if isinstance(child, ast.Import):
                    for alias in child.names:
                        module_imports.add(alias.asname or alias.name.split(".")[0])
                else:
                    for alias in child.names:
                        module_imports.add(alias.asname or alias.name)

            if isinstance(child, ast.ClassDef):
                own_methods = {
                    n.name for n in child.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))
                }
                identifiers = {child.name} | _collect_class_attributes(child) | own_methods
                blocks.append(ScopeBlock("class", child.name, child.lineno, child.end_lineno, identifiers))
                visit(child, True)
            elif isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)):
                identifiers = {child.name} | _collect_local_names(child)
                blocks.append(ScopeBlock("function", child.name, child.lineno, child.end_lineno, identifiers))
                visit(child, True)
            else:
                visit(child, inside_def)

    visit(tree, False)
    return module_imports, blocks


def chunk_file_ast(
    file_path: str,
    window_size: int = SLIDING_WINDOW_SIZE,
    stride: int = SLIDING_STRIDE,
) -> list[dict[str, Any]]:
    """Découper un fichier Python en fenêtres glissantes de lignes enrichies par l'AST.

    Chaque fenêtre hérite des identifiants de TOUS les blocs (classe et/ou
    fonction(s)) dont l'intervalle de lignes chevauche la fenêtre, plus les
    imports du module. Une fenêtre à cheval sur deux méthodes hérite ainsi
    des identifiants des deux, afin qu'une requête touchant l'une ou l'autre
    puisse retrouver ce morceau.
    """
    try:
        with open(file_path, "r", encoding="utf-8") as file:
            code = file.read()
    except OSError as error:
        print(f"Erreur de lecture de {file_path}: {error}")
        return []

    try:
        module_imports, blocks = build_scope_map(code)
    except SyntaxError as error:
        print(f"Fichier ignoré (syntaxe invalide) {file_path}: {error}")
        return []

    lines = code.splitlines()
    if not lines:
        return []

    chunks = []
    for start in range(0, len(lines), stride):
        end = min(start + window_size, len(lines))
        raw_code = "\n".join(lines[start:end]).strip()

        if raw_code:
            line_start, line_end = start + 1, end  # lignes 1-indexées, comme node.lineno

            identifiers = set(module_imports)
            for block in blocks:
                if max(line_start, block.line_start) <= min(line_end, block.line_end):
                    identifiers |= block.identifiers

            chunks.append(
                {
                    "file_path": file_path,
                    "line_start": line_start,
                    "line_end": line_end,
                    "raw_code": raw_code,
                    "identifiers": sorted(identifiers),
                }
            )

        if end >= len(lines):
            break

    return chunks


def load_and_chunk_repo_ast(
    dir_path: str,
    window_size: int = SLIDING_WINDOW_SIZE,
    stride: int = SLIDING_STRIDE,
) -> list[dict[str, Any]]:
    """Parcourir tous les fichiers .py d'un dossier et produire les chunks enrichis par AST."""
    all_chunks: list[dict[str, Any]] = []
    for root, _, files in os.walk(dir_path):
        for filename in files:
            if filename.endswith(".py"):
                file_path = os.path.join(root, filename)
                all_chunks.extend(chunk_file_ast(file_path, window_size=window_size, stride=stride))
    return all_chunks


def _repo_fingerprint(dir_path: str) -> str:
    """Empreinte du contenu d'un dossier (chemin + date de modif + taille de chaque .py).

    Sert à invalider le cache automatiquement si un fichier source a changé,
    sans avoir à relire/hacher le contenu de chaque fichier.
    """
    entries = []
    for root, _, files in os.walk(dir_path):
        for filename in files:
            if filename.endswith(".py"):
                file_path = os.path.join(root, filename)
                stat = os.stat(file_path)
                entries.append((os.path.relpath(file_path, dir_path), stat.st_mtime_ns, stat.st_size))
    entries.sort()
    return hashlib.sha1(repr(entries).encode("utf-8")).hexdigest()


def _cache_path(dir_path: str, window_size: int, stride: int, cache_dir: str | Path) -> Path:
    safe_name = re.sub(r"[^\w.-]", "_", os.path.normpath(os.path.abspath(dir_path)))
    return Path(cache_dir) / f"{safe_name}_ws{window_size}_stride{stride}.pkl"


def load_and_chunk_repo_ast_cached(
    dir_path: str,
    window_size: int = SLIDING_WINDOW_SIZE,
    stride: int = SLIDING_STRIDE,
    cache_dir: str | Path = DEFAULT_CACHE_DIR,
) -> list[dict[str, Any]]:
    """Comme `load_and_chunk_repo_ast`, mais met le résultat en cache sur disque.

    Le cache est invalidé automatiquement si un fichier .py du dossier a été
    ajouté/modifié/supprimé depuis la dernière exécution (voir `_repo_fingerprint`),
    ou si `window_size`/`stride` changent (chaque combinaison a son propre fichier
    de cache).
    """
    cache_file = _cache_path(dir_path, window_size, stride, cache_dir)
    fingerprint = _repo_fingerprint(dir_path)

    if cache_file.exists():
        with cache_file.open("rb") as file:
            cached = pickle.load(file)
        if cached.get("fingerprint") == fingerprint:
            return cached["chunks"]

    chunks = load_and_chunk_repo_ast(dir_path, window_size=window_size, stride=stride)
    cache_file.parent.mkdir(parents=True, exist_ok=True)
    with cache_file.open("wb") as file:
        pickle.dump({"fingerprint": fingerprint, "chunks": chunks}, file)
    return chunks


if __name__ == "__main__":
    project_dir = os.path.dirname(os.path.abspath(__file__))
    demo_file = os.path.join(project_dir, "retriever.py")
    chunks = chunk_file_ast(demo_file)

    print(f"{len(chunks)} morceaux générés depuis {demo_file}\n")
    for chunk in chunks[:5]:
        print(f"--- lignes {chunk['line_start']}-{chunk['line_end']} ---")
        print("identifiants:", chunk["identifiers"])
        print()

In [ ]:
%%writefile retriever.py
import json
import re
from pathlib import Path
from typing import Any

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from ast_chunker import load_and_chunk_repo_ast_cached


def best_snippet(request, snippets):
    vectorizer = TfidfVectorizer()
    matrice_repo_snippets = vectorizer.fit_transform(snippets)
    matrice_request = vectorizer.transform([request])
    scores = cosine_similarity(matrice_request, matrice_repo_snippets)[0]
    best_index = scores.argmax()
    return snippets[best_index], scores[best_index]


def tokenize_code(code: str) -> set[str]:
    """Extraire le sac de mots (bag of words) d'un extrait de code."""
    return set(re.findall(r"\w+", code))


def jaccard_similarity(tokens_a: set[str], tokens_b: set[str]) -> float:
    """Calculer l'indice de Jaccard entre deux sacs de tokens."""
    union = tokens_a | tokens_b
    if not union:
        return 0.0
    return len(tokens_a & tokens_b) / len(union)


def retrieve_top_k_jaccard(
    incomplete_code: str,
    snippets: list[dict[str, str]],
    k: int = 10,
    exclude_task_id: str | None = None,
) -> list[dict[str, Any]]:
    """Classer les blocs de code par indice de Jaccard avec le code incomplet.

    Compare le sac de tokens du code incomplet à celui de chaque bloc
    (`{"file", "snippet"}`) et retourne les k meilleurs
    `{"file", "snippet", "score"}`, triés par score décroissant.

    `exclude_task_id`, si fourni, écarte les snippets extraits de cette
    tâche : leur `right_context` contient la suite réelle du code à
    compléter (voir `dataset.extract_repository_snippets`), donc les
    inclure dans la recherche pour cette même tâche fuiterait le
    groundtruth.
    """
    query_tokens = tokenize_code(incomplete_code)

    scored_snippets = [
        {
            "file": snippet.get("file"),
            "snippet": snippet["snippet"],
            "score": jaccard_similarity(query_tokens, tokenize_code(snippet["snippet"])),
        }
        for snippet in snippets
        if exclude_task_id is None or snippet.get("task_id") != exclude_task_id
    ]
    scored_snippets.sort(key=lambda item: item["score"], reverse=True)
    return scored_snippets[:k]


def load_repository_snippets(
    repository: str, repositories_dir: str | Path
) -> list[dict[str, str]]:
    """Charger les blocs de code sauvegardés (dataset.save_repository_snippets) d'un dépôt."""
    safe_name = re.sub(r"[^\w.-]", "_", repository)
    repository_path = Path(repositories_dir) / f"{safe_name}.jsonl"

    snippets = []
    with repository_path.open("r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                record = json.loads(line)
                snippets.append(
                    {
                        "file": record.get("file"),
                        "snippet": record["snippet"],
                        "task_id": record.get("task_id"),
                    }
                )
    return snippets


def retrieve_top_k_from_repository(
    incomplete_code: str,
    repository: str,
    repositories_dir: str | Path,
    k: int = 10,
    task_id: str | None = None,
) -> list[dict[str, Any]]:
    """Charger les blocs de code d'un dépôt puis retourner les k meilleurs (Jaccard).

    `task_id`, si fourni, exclut les snippets provenant de cette même tâche
    (voir `retrieve_top_k_jaccard`).
    """
    snippets = load_repository_snippets(repository, repositories_dir)
    return retrieve_top_k_jaccard(incomplete_code, snippets, k=k, exclude_task_id=task_id)


def retrieve_top_k_ast_jaccard(
    incomplete_code: str,
    chunks: list[dict[str, Any]],
    k: int = 10,
) -> list[dict[str, Any]]:
    """Classer des chunks AST (ast_chunker.chunk_file_ast/load_and_chunk_repo_ast)
    par indice de Jaccard entre le code incomplet et les identifiants de chaque
    fenêtre.

    Contrairement à `retrieve_top_k_jaccard`, qui compare le sac de tokens bruts
    du texte, la comparaison se fait ici contre `chunk["identifiers"]` : les
    noms de classes/fonctions/attributs/variables hérités des blocs AST
    (imports du module + tout bloc classe/fonction chevauchant la fenêtre) —
    docstrings et commentaires exclus.
    """
    query_tokens = tokenize_code(incomplete_code)

    scored_chunks = [
        {
            "file_path": chunk["file_path"],
            "line_start": chunk["line_start"],
            "line_end": chunk["line_end"],
            "raw_code": chunk["raw_code"],
            "score": jaccard_similarity(query_tokens, set(chunk["identifiers"])),
        }
        for chunk in chunks
    ]
    scored_chunks.sort(key=lambda item: item["score"], reverse=True)
    return scored_chunks[:k]


def retrieve_top_k_from_directory(
    incomplete_code: str,
    dir_path: str | Path,
    k: int = 10,
) -> list[dict[str, Any]]:
    """Découper tous les fichiers .py d'un dossier avec ast_chunker (mis en cache
    sur disque, voir `ast_chunker.load_and_chunk_repo_ast_cached`) puis retourner
    les k meilleurs chunks (Jaccard sur identifiants AST)."""
    chunks = load_and_chunk_repo_ast_cached(str(dir_path))
    return retrieve_top_k_ast_jaccard(incomplete_code, chunks, k=k)


def retrieve_top_k_for_dataset(
    records: list[dict[str, Any]],
    repositories_dir: str | Path,
    k: int = 10,
    query_lines: int | None = None,
) -> list[dict[str, Any]]:
    """Appliquer le retriever Jaccard à chaque exemple de line_completion.jsonl.

    La requête vient de `record["prompt"]` (le code incomplet), jamais du
    groundtruth ; si `query_lines` est fourni, seules les `query_lines`
    dernières lignes du prompt sont utilisées (cf. `dataset.last_lines`, S_s
    dans l'article RepoCoder). Les blocs de chaque dépôt sont tokenisés une
    seule fois (mis en cache) pour éviter de retokeniser à chaque exemple.
    """
    from dataset import last_lines

    results = []
    repository_index: dict[str, list[dict[str, Any]]] = {}

    for record in records:
        metadata = record.get("metadata", {})
        repository = metadata.get("repository")
        if not repository:
            continue

        if repository not in repository_index:
            snippets = load_repository_snippets(repository, repositories_dir)
            repository_index[repository] = [
                {
                    "file": snippet["file"],
                    "snippet": snippet["snippet"],
                    "task_id": snippet.get("task_id"),
                    "tokens": tokenize_code(snippet["snippet"]),
                }
                for snippet in snippets
            ]

        task_id = metadata.get("task_id")
        query = record["prompt"] if query_lines is None else last_lines(record["prompt"], query_lines)
        query_tokens = tokenize_code(query)
        scored_snippets = [
            {
                "file": snippet["file"],
                "snippet": snippet["snippet"],
                "score": jaccard_similarity(query_tokens, snippet["tokens"]),
            }
            for snippet in repository_index[repository]
            if snippet.get("task_id") != task_id
        ]
        scored_snippets.sort(key=lambda item: item["score"], reverse=True)

        results.append(
            {
                "task_id": metadata.get("task_id"),
                "repository": repository,
                "retrieved_chunks": scored_snippets[:k],
            }
        )

    return results


if __name__ == "__main__":
    from dataset import SLIDING_STRIDE, load_jsonl, save_jsonl

    project_dir = Path(__file__).resolve().parent
    records = load_jsonl(r"C:\Users\User\Downloads\line_completion.jsonl")
    results = retrieve_top_k_for_dataset(
        records,
        project_dir / "data" / "repositories",
        k=10,
        query_lines=SLIDING_STRIDE,
    )
    save_jsonl(results, project_dir / "data" / "retrieved_jaccard.jsonl")

    print(f"{len(results)} exemples traités et sauvegardés dans data/retrieved_jaccard.jsonl")

In [ ]:
%%writefile generator_min.py
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- copié tel quel depuis repocoder-mine/generator.py ---

_hf_model_cache: dict = {}


def call_huggingface_api(
    prompt: str,
    model: str = "Salesforce/codegen-2B-mono",
    max_new_tokens: int = 64,
    temperature: float = 0.0,
    max_prompt_tokens: int = 4096,
) -> str:
    """Générer une complétion avec un modèle Hugging Face chargé localement.

    Le modèle et le tokenizer sont chargés une seule fois par nom de modèle
    (mis en cache) — utile sur Colab où `run_experiment` appelle cette
    fonction pour chaque itération de chaque exemple.
    """
    if model not in _hf_model_cache:
        tokenizer = AutoTokenizer.from_pretrained(model)
        hf_model = AutoModelForCausalLM.from_pretrained(
            model,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto",
        )
        hf_model.eval()
        _hf_model_cache[model] = (tokenizer, hf_model)

    tokenizer, hf_model = _hf_model_cache[model]
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=max_prompt_tokens
    ).to(hf_model.device)

    with torch.no_grad():
        output_ids = hf_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=temperature if temperature > 0 else None,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

## 3. Télécharger le dataset officiel RepoCoder + les 8 vrais dépôts

Même source que celle explorée en local : `microsoft/CodeT`, sous-dossier `RepoCoder` (sparse-checkout pour ne récupérer que ce dont on a besoin, pas tout `CodeT`).

In [ ]:
!git clone --no-checkout --depth 1 https://github.com/microsoft/CodeT.git codet_src
%cd codet_src
!git sparse-checkout init --cone
!git sparse-checkout set RepoCoder
!git checkout main
%cd ..

In [ ]:
import zipfile

with zipfile.ZipFile('codet_src/RepoCoder/datasets/datasets.zip') as z:
    z.extractall('datasets_rapo')
with zipfile.ZipFile('codet_src/RepoCoder/repositories/line_and_api_level.zip') as z:
    z.extractall('repos_source')

print('Dataset et dépôts extraits.')

## 4. Charger 50 tâches (échantillon reproductible sur les 8 dépôts)

In [ ]:
import json
import random
from pathlib import Path

REPOS_DIR = Path('repos_source')
TASKS_PATH = Path('datasets_rapo/line_level_completion_1k_context_codegen.test.jsonl')
N_TRIALS = 50
SEED = 42


def load_tasks(path):
    tasks = []
    with path.open('r', encoding='utf-8') as file:
        for line in file:
            if line.strip():
                tasks.append(json.loads(line))
    return tasks


all_tasks = load_tasks(TASKS_PATH)
sampled_tasks = random.Random(SEED).sample(all_tasks, N_TRIALS)
print(f'{len(all_tasks)} tâches disponibles, {len(sampled_tasks)} échantillonnées (seed={SEED}).')

## 5. Filtre anti-fuite (identique à `test_ast_retrieval.py` en local)

Écarte, dans le fichier de la tâche elle-même, toute fenêtre qui chevauche/suit la zone déjà visible dans le prompt — équivalent de `_is_context_after_hole` de l'officiel, par position de ligne réelle puisqu'on a le vrai fichier source.

In [ ]:
import os


def filter_safe_chunks(chunks, repo_dir, fpath_tuple, context_start_lineno):
    task_file = os.path.normpath(str(repo_dir.joinpath(*fpath_tuple[1:])))
    safe = []
    for chunk in chunks:
        if os.path.normpath(chunk['file_path']) == task_file:
            chunk_end_0indexed = chunk['line_end'] - 1  # ast_chunker est 1-indexé
            if chunk_end_0indexed > context_start_lineno:
                continue
        safe.append(chunk)
    return safe

## 6. Score Exact Match / Edit Similarity

Même logique que `repocoder/compute_score.py` (officiel) : comparaison ligne à ligne, `editdistance` pour l'Edit Similarity — adapté au format `metadata.ground_truth` de ce dataset (pas le format CCEval `groundtruth`/`right_context` de `metrics.py`).

In [ ]:
import editdistance


def compute_em(target: str, prediction: str) -> int:
    target_lines = [line.strip() for line in target.splitlines() if line.strip()]
    prediction_lines = [line.strip() for line in prediction.splitlines() if line.strip()][:len(target_lines)]
    return int(target_lines == prediction_lines and len(target_lines) > 0)


def compute_es(target: str, prediction: str) -> float:
    target_lines = [line.strip() for line in target.splitlines() if line.strip()]
    target_str = '\n'.join(target_lines)
    prediction_lines = [line.strip() for line in prediction.splitlines() if line.strip()][:len(target_lines)]
    prediction_str = '\n'.join(prediction_lines)
    if not target_str and not prediction_str:
        return 1.0
    return 1 - (editdistance.eval(target_str, prediction_str) / max(len(target_str), len(prediction_str), 1))

## 7. Construction du prompt (extraits récupérés + code inachevé)

Même esprit que `build_prompt.py` de l'officiel : chaque extrait récupéré est présenté avec son chemin de fichier, du moins pertinent au plus pertinent (le plus proche du code à compléter juste avant) — **plafonné à `max_context_chars`**. Sans ce plafond, avec `k=10` le contexte récupéré peut dépasser 10 000 caractères et pousser le prompt total au-delà de `max_prompt_tokens=4096` de `call_huggingface_api` ; comme la troncature du tokenizer HuggingFace coupe par défaut la **fin** du texte, c'est le code à compléter (placé en fin de prompt) qui serait tronqué, pas le contexte — l'inverse de ce qu'on veut. Le plafond garantit que le code à compléter reste toujours intact.

In [ ]:
def build_prompt(unfinished_code, retrieved_chunks, repo_dir, max_context_chars=6000):
    blocks = []
    total_chars = 0
    for chunk in retrieved_chunks:  # déjà trié du meilleur au moins bon
        rel_path = os.path.relpath(chunk['file_path'], repo_dir)
        block = f"# the below code fragment can be found in: {rel_path}\n{chunk['raw_code']}\n"
        if total_chars + len(block) > max_context_chars:
            break
        blocks.append(block)
        total_chars += len(block)
    context = '\n'.join(reversed(blocks))  # le plus pertinent juste avant le code
    return f"{context}\n{unfinished_code}"

## 8. Boucle des 50 essais

Charge le modèle HuggingFace une seule fois (mis en cache par `generator_min.call_huggingface_api`), puis pour chaque tâche : découpe AST du dépôt (mise en cache sur disque, un seul découpage par dépôt), retrieval, prompt, génération, score.

In [ ]:
from ast_chunker import load_and_chunk_repo_ast_cached
from retriever import retrieve_top_k_ast_jaccard
from generator_min import call_huggingface_api

# Le dataset line_level_completion_*_context_codegen est calibré avec CodeGenTokenizer
# (Salesforce/codegen-6B-mono, cf. repocoder/utils.py) : on utilise donc un modèle
# CodeGen-mono, cohérent avec le papier — pas un modèle d'une autre famille.
# 'Salesforce/codegen-6B-mono' = fidèle au papier (~12 Go en fp16, GPU type A100 recommandé)
# 'Salesforce/codegen-350M-mono' = repli rapide pour un test de plomberie
MODEL_NAME = 'Salesforce/codegen-2B-mono'  # bon compromis mémoire/fidélité sur un T4 gratuit

chunk_cache = {}
results = []

for i, task in enumerate(sampled_tasks, start=1):
    metadata = task['metadata']
    repo = metadata['task_id'].split('/')[0]
    repo_dir = REPOS_DIR / repo

    if repo not in chunk_cache:
        print(f'[{i}/{len(sampled_tasks)}] Découpage AST de {repo} ...')
        chunk_cache[repo] = load_and_chunk_repo_ast_cached(str(repo_dir))

    safe_chunks = filter_safe_chunks(
        chunk_cache[repo], repo_dir, metadata['fpath_tuple'], metadata['context_start_lineno']
    )
    retrieved = retrieve_top_k_ast_jaccard(task['prompt'], safe_chunks, k=10)
    prompt = build_prompt(task['prompt'], retrieved, repo_dir)

    completion = call_huggingface_api(prompt, model=MODEL_NAME, max_new_tokens=64)

    em = compute_em(metadata['ground_truth'], completion)
    es = compute_es(metadata['ground_truth'], completion)
    results.append({
        'task_id': metadata['task_id'],
        'ground_truth': metadata['ground_truth'],
        'completion': completion,
        'exact_match': em,
        'edit_similarity': es,
    })

    print(f"[{i}/{len(sampled_tasks)}] {metadata['task_id']:<35} EM={em}  ES={es:.2f}")


## 9. Résultats agrégés

In [ ]:
import json

with open('results_50_trials.jsonl', 'w', encoding='utf-8') as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

exact_matches = sum(r['exact_match'] for r in results)
avg_es = sum(r['edit_similarity'] for r in results) / len(results)

print(f'Essais : {len(results)}')
print(f'Exact Match : {exact_matches}/{len(results)} ({100 * exact_matches / len(results):.1f}%)')
print(f'Edit Similarity moyenne : {avg_es:.3f}')
print()
print('Résultats détaillés sauvegardés dans results_50_trials.jsonl')